# 3장. 데이터의 첫인상 읽기

이 노트북은 강의안 `book/chapters/ch03_data_first_impression.md`를 따라가며 직접 실행해 보는 실습용 자료입니다.

이번 장의 목표는 멋진 분석 결과를 바로 만드는 것이 아니라, 분석 전에 데이터가 어떤 모양인지 차분히 확인하는 습관을 만드는 것입니다. 코드를 한 셀씩 실행하면서 출력 결과를 보고, 바로 아래 설명과 질문에 답해 보세요.


## 0. 이번 장에서 확인할 것

데이터를 처음 열었을 때는 다음 질문에 답할 수 있어야 합니다.

- 데이터 파일은 몇 개인가?
- 각 파일은 어떤 역할을 하는가?
- 각 파일은 몇 행, 몇 열로 구성되어 있는가?
- 어떤 컬럼이 있고, 각 컬럼은 어떤 의미를 가지는가?
- 숫자, 문자, 날짜 컬럼은 무엇인가?
- 비어 있는 값이나 중복된 값은 없는가?
- 여러 파일을 연결할 수 있는 기준 컬럼은 무엇인가?
- LLM이 설명한 데이터 구조가 실제 데이터와 일치하는가?


![CSV 파일을 pandas DataFrame으로 불러오는 흐름](../book/assets/images/ch03/ch03_csv_to_dataframe_flow.svg)

CSV 파일은 텍스트 파일이지만, pandas로 불러오면 행과 열을 가진 `DataFrame`으로 다룰 수 있습니다.


## 1. 실습 준비

먼저 필요한 패키지를 불러오고, 프로젝트 폴더와 데이터 폴더를 찾습니다. 노트북을 `notebooks` 폴더에서 실행해도 되고, 프로젝트 루트에서 실행해도 되도록 `find_project_root()` 함수를 사용합니다.


In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def find_project_root(start: Path) -> Path:
    """현재 위치에서 위로 올라가며 data/raw 폴더가 있는 프로젝트 루트를 찾습니다."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "book").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 노트북을 저장소 안에서 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)


프로젝트 루트: G:\내 드라이브\dev_GD\llm-data-analysis-study
데이터 폴더: G:\내 드라이브\dev_GD\llm-data-analysis-study\data\raw


## 2. 데이터 파일이 있는지 확인하기

파일 경로 오류는 초보자가 가장 자주 만나는 오류입니다. 데이터를 불러오기 전에 필요한 CSV 파일이 실제로 있는지 먼저 확인합니다.


In [3]:
expected_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

file_check = pd.DataFrame({
    "file": expected_files,
    "path": [str(DATA_DIR / filename) for filename in expected_files],
    "exists": [(DATA_DIR / filename).exists() for filename in expected_files],
})

file_check


,file,path,exists
0,customers.csv,G:\내 드라이브\dev_GD\llm-data-analysis-study\data\...,True
1,products.csv,G:\내 드라이브\dev_GD\llm-data-analysis-study\data\...,True
2,orders.csv,G:\내 드라이브\dev_GD\llm-data-analysis-study\data\...,True
3,order_items.csv,G:\내 드라이브\dev_GD\llm-data-analysis-study\data\...,True


`exists`가 모두 `True`이면 다음 단계로 진행할 수 있습니다.

하나라도 `False`라면 데이터가 아직 생성되지 않았을 수 있습니다. 그 경우 터미널에서 아래 명령을 실행해 샘플 데이터를 생성합니다.

```bash
python scripts/generate_sample_data.py
```


## 3. CSV 파일을 DataFrame으로 불러오기

이제 4개의 CSV 파일을 pandas `DataFrame`으로 불러옵니다. 각 변수 이름은 파일 이름과 비슷하게 맞춰 두면 이후 코드를 읽기 쉽습니다.


In [4]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

print("customers:", type(customers))
print("products:", type(products))
print("orders:", type(orders))
print("order_items:", type(order_items))


customers: <class 'pandas.DataFrame'>
products: <class 'pandas.DataFrame'>
orders: <class 'pandas.DataFrame'>
order_items: <class 'pandas.DataFrame'>


분석할 데이터셋이 여러 개일 때는 딕셔너리로 묶어 두면 반복 점검을 하기 편합니다.


In [5]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

list(datasets.keys())


['customers', 'products', 'orders', 'order_items']

## 4. 각 파일의 역할 이해하기

이번 과정에서 사용하는 데이터는 가상의 온라인 쇼핑몰 운영 데이터입니다.

| 파일 | 역할 | 먼저 확인할 것 |
| --- | --- | --- |
| `customers.csv` | 고객 정보 | 고객 수, 연령, 성별, 지역, 가입일 |
| `products.csv` | 상품 정보 | 상품 수, 카테고리, 가격 |
| `orders.csv` | 주문 정보 | 주문 수, 주문일, 결제수단, 주문상태 |
| `order_items.csv` | 주문 상세 정보 | 주문별 상품, 수량, 단가 |

처음에는 파일을 합치지 말고, 각 파일을 따로 살펴보는 것이 좋습니다.


![pandas DataFrame 구조 예시](../book/assets/images/ch03/ch03_dataframe_structure.svg)

DataFrame은 행(row)과 열(column)로 구성됩니다. `shape`, `head()`, `columns`, `info()` 같은 기본 도구를 사용해 구조를 확인합니다.


## 5. 데이터 크기 확인하기

`shape`는 데이터의 행과 열 개수를 알려 줍니다.

- 앞 숫자: 행 개수
- 뒤 숫자: 열 개수

예를 들어 `(150, 6)`은 150행 6열이라는 뜻입니다.


In [5]:
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)


In [6]:
shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    for name, df in datasets.items()
])

shape_summary


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


### 생각해 보기

- 가장 행이 많은 데이터셋은 무엇인가요? 
order_items
- `order_items`가 `orders`보다 행이 많다면, 그 이유는 무엇일까요?
orders는 주문 자체를 나타내고 order_items는 주문 안에 들어 있는 개별 상품을 나타내는 값이기 때문에 주문 1건에 여러 상품이 들어가면 oders보다 더 많은 행이 나올 수 있습니다.
- 분석 보고서에 데이터 규모를 설명한다면 어떤 문장으로 쓸 수 있을까요? 
현재 사용하는 CSV 데이터는 customers 150건, products 100건, orders 300건, order_items 764건으로 구성되어 있습니다.


## 6. 데이터 앞부분과 마지막 부분 보기

`head()`는 앞부분 5행을 보여 줍니다. 컬럼명이 예상과 맞는지, 값의 형태가 자연스러운지 빠르게 확인할 때 사용합니다.


In [7]:
customers.head()


,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-06-19
1,2,김정호,F,32,대구,2025-11-02
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
4,5,이예원,F,19,부산,2024-09-13


In [8]:
products.head()


,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


In [9]:
orders.head()


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-05-07,card,completed
1,2,77,2025-07-23,naver_pay,cancelled
2,3,138,2025-11-19,bank_transfer,cancelled
3,4,57,2026-01-30,kakao_pay,cancelled
4,5,125,2025-12-21,card,cancelled


In [10]:
order_items.head()


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


앞부분만 보고 전체 데이터가 정상이라고 판단하기는 어렵습니다. `tail()`로 마지막 부분도 확인해 봅니다.


In [11]:
customers.tail()


,customer_id,name,gender,age,city,signup_date
145,146,김숙자,M,61,성남,2025-12-23
146,147,이정남,M,19,부산,2025-02-11
147,148,오도현,M,29,고양,2026-06-15
148,149,김정자,M,20,부산,2024-10-19
149,150,조미영,M,40,대전,2025-12-04


### 생각해 보기

`head()`와 `tail()`을 보면서 아래를 확인해 보세요.

- 날짜처럼 보이는 컬럼이 있나요? customer 의 signup_date와 orders의 order_data가 날짜로 보입니다.
- 숫자처럼 보이는 컬럼이 있나요? customer의 age, products의 price, order_items의 quantity/unit_price 가 숫자형 데이터로 보입니다.
- ID처럼 보이는 컬럼이 있나요? customers의 customer_id, products의 product_id, orders의 order_id/customer_id, order_items의 order_item_id 가 ID 처럼 보입니다.
- 사람이 직접 읽을 수 있는 이름이나 범주 값이 있나요? customers의 name/gender/city , products에는 product_name/category, orders에는 payment_method, order_status 같이 사람이 의미를 해석할수 있는 값이 포함되어 있습니다. 


## 7. 컬럼명 확인하기

컬럼명은 코드 작성에서 매우 중요합니다. 실제 컬럼명이 `customer_id`인데 LLM이나 사람이 `cust_id`라고 쓰면 코드는 실행되지 않습니다.


In [12]:
for name, df in datasets.items():
    print(f"[{name}]")
    print(list(df.columns))
    print()


[customers]
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
['product_id', 'product_name', 'category', 'price']

[orders]
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [13]:
column_summary = pd.DataFrame([
    {
        "dataset": name,
        "column_count": len(df.columns),
        "column_names": ", ".join(df.columns),
    }
    for name, df in datasets.items()
])

column_summary


,dataset,column_count,column_names
0,customers,6,"customer_id, name, gender, age, city, signup_date"
1,products,4,"product_id, product_name, category, price"
2,orders,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,5,"order_item_id, order_id, product_id, quantity,..."


### 생각해 보기

- 고객을 구분하는 컬럼은 무엇인가요? customers의 customer_id가 고객을 구분하기 위한 ID 칼럼으로 보입니다.
- 주문을 구분하는 컬럼은 무엇인가요? orders의 order_id가 각 주문을 구분하는 ID 칼럼으로 보입니다.
- 상품을 구분하는 컬럼은 무엇인가요? products의 product_id가 상품을 구분하는 칼럼으로 보입니다.
- 여러 파일을 연결할 때 사용할 수 있을 것 같은 컬럼은 무엇인가요?
위에서 언급한 customer_id, order_id, product_id가 여러 파일들을 연결할 때 사용할 수 있는 공통적인 컬럼으로 보입니다.
orders의 customer_id -> customers의 customer_id
orders_items 의 order_id -> orders 의 order_id
order_items의 product_id -> products 의 product_id 와 같이 고객, 주문, order_items, 상품 데이터를 ID 기준으로 연결하여 분석이 가능할 것으로 판단됩니다. 


## 8. 데이터 타입 확인하기

`info()`는 컬럼별 데이터 타입과 비어 있지 않은 값의 개수를 보여 줍니다.

특히 날짜처럼 보이지만 `object`로 저장된 컬럼을 주의해서 봅니다. pandas에서 `object`는 보통 문자열 또는 여러 타입이 섞인 컬럼일 때 나타납니다.


In [14]:
customers.info()


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB


In [15]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    df.info()



===== customers =====
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB

===== products =====
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 6.0 KB

===== orders =====
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns)

In [16]:
dtype_summary = pd.concat(
    [df.dtypes.rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

dtype_summary


,customers,products,orders,order_items
customer_id,int64,,int64,
name,str,,,
gender,str,,,
age,int64,,,
city,str,,,
signup_date,str,,,
product_id,,int64,,int64
product_name,,str,,
category,,str,,
price,,int64,,


### 생각해 보기

- 숫자형 컬럼은 어떤 것들이 있나요? 데이터 타입으로는 각 csv별로 아래 칼럼이 int64로 숫자형으로 저장되어 있지만 이중에서 customer_id,product_id,order_item_id는 계산하거나 크기를 구하기 위한 값이 아니라 다른 데이터셋이랑 연결하기 위한 ID 이다.
customers : customer_id, age
products : product_id, price
orders : order_id, customer_id
order_items : order_item_id, order_id, product_id, quantity, unit_price

- 문자형 컬럼은 어떤 것들이 있나요?
데이터 타입으로 각 csv 별로 아래 칼럼이 str로 문자형으로 지정되어 있다.
customers : name, gender, city, signup_date
products : product_name, category
orders : order_date, payment_method, order_status

- 날짜처럼 보이지만 아직 문자열일 가능성이 있는 컬럼은 무엇인가요?
signup_date와 order_date가 날짜 형식의 값을 가지고 있지만 현재 데이터 타입은 모두 str로 확인되었다.  
따라서 분석을 수행하기 전에는 실제 날짜 타입인으로 변환하고 변환 과정에서 오류가 발생하는 값이 있는지도 확인할 필요가 있다.

![데이터 구조 점검 흐름도](../book/assets/images/ch03/ch03_data_check_flow.svg)

데이터 구조 점검은 파일 확인, 로드, 크기 확인, 컬럼 확인, 타입 확인, 결측치 확인, 중복 확인, 키 관계 확인 순서로 진행하면 좋습니다.


## 9. 결측치 확인하기

결측치는 값이 비어 있는 상태입니다. 결측치가 있으면 평균, 비율, 그룹별 집계 결과가 달라질 수 있습니다.

`isna().sum()`은 컬럼별 결측치 개수를 계산합니다.


In [17]:
customers.isna().sum()


customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

In [18]:
missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("").astype(str)

missing_summary


,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


In [19]:
missing_rate_summary = pd.concat(
    [(df.isna().mean() * 100).round(2).rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

missing_rate_summary


,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


### 결측치 해석 팁

결측치가 있다고 해서 무조건 삭제하는 것은 아닙니다.

| 처리 방법 | 설명 |
| --- | --- |
| 행 제외 | 결측치가 있는 행을 분석에서 제외합니다. |
| 대표값 대체 | 평균, 중앙값, 최빈값 등으로 채웁니다. |
| 별도 범주 처리 | `Unknown` 같은 범주로 표시합니다. |
| 컬럼 제외 | 분석 목적에 맞지 않는 컬럼은 사용하지 않습니다. |
| 원인 확인 | 수집 과정에서 문제가 있었는지 확인합니다. |


## 10. 중복 데이터 확인하기

중복은 같은 행이나 같은 ID가 반복되는 상태입니다.

단, 모든 중복이 오류는 아닙니다. 예를 들어 `order_items`에서는 한 주문에 여러 상품이 들어갈 수 있으므로 같은 `order_id`가 여러 번 나올 수 있습니다.


In [20]:
duplicate_rows = pd.DataFrame([
    {
        "dataset": name,
        "duplicated_rows": df.duplicated().sum(),
    }
    for name, df in datasets.items()
])

duplicate_rows


,dataset,duplicated_rows
0,customers,0
1,products,0
2,orders,0
3,order_items,0


In [21]:
id_duplicate_checks = pd.DataFrame([
    {
        "check": "customers.customer_id",
        "duplicated_count": customers["customer_id"].duplicated().sum(),
        "interpretation": "0이어야 고객 ID가 유일합니다.",
    },
    {
        "check": "products.product_id",
        "duplicated_count": products["product_id"].duplicated().sum(),
        "interpretation": "0이어야 상품 ID가 유일합니다.",
    },
    {
        "check": "orders.order_id",
        "duplicated_count": orders["order_id"].duplicated().sum(),
        "interpretation": "0이어야 주문 ID가 유일합니다.",
    },
    {
        "check": "order_items.order_id",
        "duplicated_count": order_items["order_id"].duplicated().sum(),
        "interpretation": "한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.",
    },
])

id_duplicate_checks


,check,duplicated_count,interpretation
0,customers.customer_id,0,0이어야 고객 ID가 유일합니다.
1,products.product_id,0,0이어야 상품 ID가 유일합니다.
2,orders.order_id,0,0이어야 주문 ID가 유일합니다.
3,order_items.order_id,464,한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.


### 생각해 보기

- `customers.customer_id` 중복과 `order_items.order_id` 중복은 왜 의미가 다를까요?
customers.customer_id는 고객 한명을 식별하기 위한 ID이므로 각 고객마다 고유한 값이 있어야 한다. 따라서 이 컬럼에서 중복이 발생하면 서로 다른 행이 같은 고객 ID를 가지고 있을 가능성이 있어서 데이터 품질을 의심해 봐야한다.
반면 order_items.order_id는 각 주문한 아이템이 어느 주문에 속하는지를 나타내는 연결용 ID이다. 하나의 주문에 여러 상품이 포함될 수 있으므로 하나의 order_id가 여러 order_items 행에서 반복되는 것은 정상적인 데이터 구조이다.

- 중복 개수만 보고 삭제하면 위험한 이유는 무엇일까요?
중복은 상황에 따라서 정상일 수도 있고 오류일 수도 있기 때문이다.
customers.customer_id 처럼 고유해야하는 식별 ID에서 중복이 발생하면 오류 확인이 필요하지만 order_items.order_id처럼 여러행이 하나의 주문을 참조하는 구조라면 중복되는 것이 정상이다.

## 11. 숫자형 컬럼 기본 통계 확인하기

`describe()`는 숫자형 컬럼의 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 보여 줍니다.

최솟값이나 최댓값이 지나치게 이상하면 데이터 오류나 이상치 가능성을 의심할 수 있습니다.


In [22]:
customers.describe()


,customer_id,age
count,150.000000,150.000000
mean,75.500000,42.086667
std,43.445368,15.613166
min,1.000000,19.000000
25%,38.250000,29.000000
50%,75.500000,40.000000
75%,112.750000,57.000000
max,150.000000,69.000000


In [23]:
products[["price"]].describe()


,price
count,100.000000
mean,110040.000000
std,56433.910574
min,5000.000000
25%,65750.000000
50%,112000.000000
75%,161000.000000
max,200000.000000


In [24]:
order_items[["quantity", "unit_price"]].describe()


,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


숫자가 문자열로 저장된 경우도 있습니다. 예를 들어 `"10,000"`처럼 쉼표가 포함된 문자열은 바로 계산하기 어렵습니다.


In [25]:
price_text = pd.Series(["10,000", "25,500", "3000", "확인필요"])
price_number = pd.to_numeric(
    price_text.str.replace(",", "", regex=False),
    errors="coerce",
)

pd.DataFrame({
    "original": price_text,
    "converted": price_number,
})


,original,converted
0,"10,000",10000.0
1,"25,500",25500.0
2,3000,3000.0
3,확인필요,NaN


`errors="coerce"`는 숫자로 바꿀 수 없는 값을 `NaN`으로 처리합니다. 변환 후에는 새로 생긴 결측치가 있는지도 확인해야 합니다.


## 12. 범주형 컬럼 고유값 확인하기

문자형 또는 범주형 컬럼은 고유값 개수와 빈도를 확인합니다. 예를 들어 지역, 성별, 카테고리, 주문 상태 같은 컬럼은 `value_counts()`로 분포를 볼 수 있습니다.


In [26]:
customers["city"].value_counts().head(10)


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

In [27]:
products["category"].value_counts()


category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

In [28]:
orders["order_status"].value_counts()


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

In [29]:
categorical_summary = pd.DataFrame([
    {"dataset": "customers", "column": "city", "unique_count": customers["city"].nunique()},
    {"dataset": "customers", "column": "gender", "unique_count": customers["gender"].nunique()},
    {"dataset": "products", "column": "category", "unique_count": products["category"].nunique()},
    {"dataset": "orders", "column": "payment_method", "unique_count": orders["payment_method"].nunique()},
    {"dataset": "orders", "column": "order_status", "unique_count": orders["order_status"].nunique()},
])

categorical_summary


,dataset,column,unique_count
0,customers,city,10
1,customers,gender,2
2,products,category,7
3,orders,payment_method,4
4,orders,order_status,3


### 생각해 보기

- 특정 값에 데이터가 지나치게 몰려 있나요?
city는 10개 지역에 걸쳐서 11~21정도로 분포하여 특정 지역에 집중되어 있다고 보기는 어렵다.
category는 최대 19 최소7건으로 어느정도 차이는 있지만 한 카테고리가 대부분을 차지하는 정도는 아니다.
하지만 order_status는 completed가 184건으로 약61.3%를 차지하여 상대적으로 높은 비중을 보이고 있어 completed에 집중된 분포로 볼 수 있다.
- 오타나 표기 차이처럼 보이는 값이 있나요?
위에서 확인한 city, category, order_status는 동일한 의미가 서로다른 표기로 입력된 것으로 보이는 값이나 뚜렷한 오타는 확인되지 않았다. unique_count와 수와 value_counts로 확인한 갯수가 동일한것으로 확인하였다.
- 나중에 그룹별 분석 기준으로 쓰기 좋은 컬럼은 무엇인가요?
city, gender, category, payment_method, order_status가 그룹별 분석 기준으로 활용하기 좋다.  
예를 들어 지역별 고객 수, 성별 구매 특성, 상품 카테고리별 판매량, 결제수단별 주문 특성 등을 비교할 수 있다.  


## 13. 날짜 컬럼 확인하기

날짜 컬럼은 월별, 요일별, 기간별 분석에 자주 사용됩니다.

하지만 CSV에서 읽어온 날짜는 처음에는 문자열(`object`)일 수 있습니다. `pd.to_datetime()`으로 날짜 타입으로 바꿔야 날짜 계산을 안전하게 할 수 있습니다.


In [30]:
orders["order_date"].head()


0    2026-05-07
1    2025-07-23
2    2025-11-19
3    2026-01-30
4    2025-12-21
Name: order_date, dtype: str

In [31]:
print("변환 전 타입:", orders["order_date"].dtype)

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())


변환 전 타입: str
변환 후 타입: datetime64[us]
날짜 변환 실패 건수: 0
가장 빠른 주문일: 2025-07-09 00:00:00
가장 최근 주문일: 2026-07-08 00:00:00


In [32]:
orders.assign(
    order_year=orders["order_date"].dt.year,
    order_month=orders["order_date"].dt.month,
    order_day_name=orders["order_date"].dt.day_name(),
).head()


,order_id,customer_id,order_date,payment_method,order_status,order_year,order_month,order_day_name
0,1,123,2026-05-07,card,completed,2026,5,Thursday
1,2,77,2025-07-23,naver_pay,cancelled,2025,7,Wednesday
2,3,138,2025-11-19,bank_transfer,cancelled,2025,11,Wednesday
3,4,57,2026-01-30,kakao_pay,cancelled,2026,1,Friday
4,5,125,2025-12-21,card,cancelled,2025,12,Sunday


### 생각해 보기

- 데이터는 어느 기간을 포함하고 있나요?
2025-07-09부터 2026-07-08까지의 기간을 포함하고 있습니다.
- 월별 매출 분석을 하기에 충분한 기간인가요?
약 1년의 데이터가 있으므로 월별 매출의 변화를 확인하는 기본적인 분석은 가능하다고 판단됩니다.
- 날짜 변환 실패 건수가 0보다 크다면 무엇을 확인해야 할까요?
먼저 변환에 실패한 원본값을 직접 확인해야하고 날짜 형식이 다르거나 날짜 이외의 문자가 포함되었는지 아니면 오타가 포함되었는지 확인할 필요가 있습니다.
원본값과 발생원인 확인한 후 수정 또는 제외여부를 결정하는 필요합니다.


## 14. 여러 파일의 관계 확인하기

온라인 쇼핑몰 데이터는 고객, 상품, 주문, 주문 상세 데이터가 서로 연결되어야 분석할 수 있습니다.

| 연결 관계 | 의미 |
| --- | --- |
| `customers.customer_id` ↔ `orders.customer_id` | 어떤 고객이 주문했는지 연결합니다. |
| `orders.order_id` ↔ `order_items.order_id` | 주문과 주문 상세를 연결합니다. |
| `products.product_id` ↔ `order_items.product_id` | 주문 상세와 상품 정보를 연결합니다. |


![4개 CSV 파일 간 키 관계도](../book/assets/images/ch03/ch03_csv_key_relationships.svg)


In [33]:
invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]
invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]

relationship_check = pd.DataFrame([
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "invalid_rows": len(invalid_customers),
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "invalid_rows": len(invalid_orders),
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "invalid_rows": len(invalid_products),
    },
])

relationship_check


,relationship,invalid_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0


`invalid_rows`가 모두 0이면 샘플 데이터에서는 기본적인 연결 관계가 유지되고 있다고 볼 수 있습니다. 0보다 큰 값이 있다면 어느 파일에서 기준 ID가 빠져 있는지 먼저 확인해야 합니다.


## 15. 간단한 병합으로 관계 확인하기

키 관계가 맞는지 확인한 뒤에는 데이터를 병합해 볼 수 있습니다. 아래 코드는 주문 상세(`order_items`)에 상품 정보(`products`)를 붙이고, 각 행의 금액을 계산합니다.


In [34]:
order_items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
)

order_items_with_products["line_amount"] = (
    order_items_with_products["quantity"] * order_items_with_products["unit_price"]
)

order_items_with_products.head()


,order_item_id,order_id,product_id,quantity,unit_price,product_name,category,price,line_amount
0,1,1,100,3,102000,도서 상품 100,도서,102000,306000
1,2,1,87,5,25000,도서 상품 087,도서,25000,125000
2,3,1,7,3,142000,도서 상품 007,도서,142000,426000
3,4,1,9,3,193000,스포츠 상품 009,스포츠,193000,579000
4,5,2,72,4,189000,뷰티 상품 072,뷰티,189000,756000


In [35]:
category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales


,category,line_amount
3,스포츠,50174000
1,뷰티,47551000
5,전자기기,41003000
2,생활용품,34839000
4,식품,33597000
0,도서,24645000
6,패션,23801000


이 결과는 본격적인 EDA가 아니라, 데이터 관계가 실제로 연결되는지 확인하는 작은 점검입니다. 분석 결론을 내리기 전에 결측치, 주문 상태, 취소 주문 처리 기준 등을 더 확인해야 합니다.


## 16. 반복 점검을 함수로 정리하기

여러 데이터셋에 같은 점검을 반복할 때는 함수로 정리하면 편합니다.


In [36]:
def check_data_overview(name: str, df: pd.DataFrame) -> None:
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(list(df.columns))
    print("\ndtypes:")
    print(df.dtypes)
    print("\nmissing values:")
    print(df.isna().sum())
    print("\nduplicated rows:", df.duplicated().sum())


check_data_overview("customers", customers)


===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0


In [37]:
for name, df in datasets.items():
    check_data_overview(name, df)
    print()


===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0

===== products =====
shape: (100, 4)

columns:
['product_id', 'product_name', 'category', 'price']

dtypes:
product_id      int64
product_name      str
category          str
price           int64
dtype: object

missing values:
product_id      0
product_name    0
category        0
price           0
dtype: int64

duplicated rows: 0

===== orders =====
shape: (300, 5)

columns:
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

dtypes:
order_id                   int64
customer_id                int64
order_date        datetime64[us]
payment_method          

![Jupyter Notebook 데이터 구조 점검 결과 화면 예시](../book/assets/images/ch03/ch03_jupyter_data_overview_result.svg)


## 17. LLM에게 데이터 구조를 설명시키는 법

LLM에게 원본 데이터를 그대로 붙여 넣는 것은 피하는 것이 좋습니다. 대신 아래처럼 구조 요약만 전달합니다.

- 파일명
- 컬럼명
- 행과 열 개수
- 데이터 타입
- 결측치 개수
- 중복 여부
- 파일 간 키 관계


In [38]:
llm_dataset_summary = shape_summary.merge(column_summary, on="dataset")
llm_dataset_summary


,dataset,rows,columns,column_count,column_names
0,customers,150,6,6,"customer_id, name, gender, age, city, signup_date"
1,products,100,4,4,"product_id, product_name, category, price"
2,orders,300,5,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,764,5,5,"order_item_id, order_id, product_id, quantity,..."


In [39]:
for _, row in llm_dataset_summary.iterrows():
    print(f"- {row['dataset']}: {row['rows']}행 {row['columns']}열")
    print(f"  컬럼: {row['column_names']}")


- customers: 150행 6열
  컬럼: customer_id, name, gender, age, city, signup_date
- products: 100행 4열
  컬럼: product_id, product_name, category, price
- orders: 300행 5열
  컬럼: order_id, customer_id, order_date, payment_method, order_status
- order_items: 764행 5열
  컬럼: order_item_id, order_id, product_id, quantity, unit_price


### 데이터 구조 설명 요청 예시

아래 프롬프트는 LLM에게 붙여 넣을 수 있는 예시입니다. 실제 데이터 전체가 아니라 구조 정보만 포함합니다.


In [40]:
prompt = f"""
온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
{llm_dataset_summary[['dataset', 'rows', 'columns', 'column_names']].to_string(index=False)}

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.
"""

print(prompt)



온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
    dataset  rows  columns                                                    column_names
  customers   150        6               customer_id, name, gender, age, city, signup_date
   products   100        4                       product_id, product_name, category, price
     orders   300        5 order_id, customer_id, order_date, payment_method, order_status
order_items   764        5       order_item_id, order_id, product_id, quantity, unit_price

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.



## 18. LLM 답변 검증 연습

LLM이 다음과 같이 답했다고 가정해 봅니다.

> 고객 데이터에 age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 됩니다.

이 답변은 그럴듯하지만 충분히 안전하지 않습니다. 아래 내용을 직접 확인해야 합니다.

- `age` 컬럼이 실제로 존재하는가?
- `age` 컬럼에 결측치나 이상치가 있는가?
- 고객 데이터와 주문 데이터가 `customer_id`로 연결되는가?
- 매출을 계산하려면 주문 상세와 상품 또는 단가 정보가 필요한가?
- 취소 주문을 포함할지 제외할지 기준이 있는가?


In [41]:
validation_check = pd.DataFrame([
    {
        "question": "customers에 age 컬럼이 있는가?",
        "result": "age" in customers.columns,
    },
    {
        "question": "age 결측치 개수는?",
        "result": customers["age"].isna().sum() if "age" in customers.columns else "컬럼 없음",
    },
    {
        "question": "orders.customer_id가 customers.customer_id와 연결되는가?",
        "result": len(invalid_customers) == 0,
    },
    {
        "question": "매출 계산에 필요한 quantity와 unit_price가 있는가?",
        "result": {"quantity", "unit_price"}.issubset(order_items.columns),
    },
])

validation_check


,question,result
0,customers에 age 컬럼이 있는가?,True
1,age 결측치 개수는?,0
2,orders.customer_id가 customers.customer_id와 연결되는가?,True
3,매출 계산에 필요한 quantity와 unit_price가 있는가?,True


## 19. 이번 장 점검 체크리스트

| 점검 항목 | 확인 |
| --- | --- |
| 필요한 CSV 파일이 모두 존재하는가? | V |
| 각 데이터셋의 행과 열 개수를 확인했는가? | V |
| 컬럼명이 예상과 일치하는가? | V |
| 날짜 컬럼의 데이터 타입을 확인했는가? | V |
| 숫자 컬럼이 실제 숫자형으로 저장되어 있는가? | V |
| 결측치가 있는 컬럼을 확인했는가? | V |
| 중복 데이터가 있는지 확인했는가? | V |
| 주요 ID 컬럼의 중복 여부를 확인했는가? | V |
| 여러 파일을 연결할 키 컬럼을 확인했는가? | V |
| 파일 간 키 관계가 실제로 연결 가능한지 확인했는가? | V |
| LLM에 원본 데이터 대신 구조 요약만 입력했는가? | V |
| LLM이 제안한 설명을 실제 데이터와 비교해 검증했는가? | V |


## 20. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.
2. 각 파일의 결측치 개수와 결측치 비율을 확인하세요.
3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.
4. `order_items`와 `products`를 병합해 카테고리별 주문 금액을 계산하세요.
5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.
6. LLM이 만든 분석 아이디어가 실제 컬럼과 키 관계에 맞는지 검증하세요.


In [ ]:
# 과제 풀이 공간입니다.
# 필요한 코드를 직접 작성해 보세요.
# 1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.
shape_result = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in datasets.items()
])

shape_result



In [8]:
# 2. 각 CSV 파일의 컬럼 이름을 확인하고, 데이터 타입과 결측치 개수를 함께 정리하세요.
missing_result = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        missing_rate = df[column].isna().mean() * 100

        missing_result.append({
            "dataset": name,
            "column": column,
            "missing_count": missing_count,
            "missing_rate(%)": round(missing_rate, 2)
        })

missing_result = pd.DataFrame(missing_result)

missing_result

,dataset,column,missing_count,missing_rate(%)
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


In [9]:
#3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

date_result = pd.DataFrame({
    "item": [
        "order_date dtype",
        "날짜 변환 실패 건수",
        "가장 빠른 주문일",
        "가장 최근 주문일"
    ],
    "result": [
        str(orders["order_date"].dtype),
        orders["order_date"].isna().sum(),
        orders["order_date"].min(),
        orders["order_date"].max()
    ]
})

date_result

,item,result
0,order_date dtype,datetime64[us]
1,날짜 변환 실패 건수,0
2,가장 빠른 주문일,2025-07-09 00:00:00
3,가장 최근 주문일,2026-07-08 00:00:00


In [14]:
# 4. order_items와 products를 병합해 카테고리별 주문 금액 계산

order_items_products = order_items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left"
)

# 주문 상세 한 행의 금액 계산
order_items_products["line_amount"] = (
    order_items_products["quantity"]
    * order_items_products["unit_price"]
)

# 카테고리별 주문 금액 합계
category_amount = (
    order_items_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

print(category_amount.to_string(index=False))


category  line_amount
     스포츠     50174000
      뷰티     47551000
    전자기기     41003000
    생활용품     34839000
      식품     33597000
      도서     24645000
      패션     23801000


#5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.

데이터 분석을 위해 4가지 csv파일의 현재까지 직접 확인한 데이터 구조를 공유하겠습니다.

1. customers
- 크기: 150행 × 6열
- 컬럼: customer_id, name, gender, age, city, signup_date
- customer_id 중복: 0
- 결측치: 없음
- signup_date는 날짜 형태로 보이지만 원본에서는 문자열로 저장되어 있음

2. products
- 크기: 100행 × 4열
- 컬럼: product_id, product_name, category, price
- product_id 중복: 0
- 결측치: 없음

3. orders
- 크기: 300행 × 5열
- 컬럼: order_id, customer_id, order_date, payment_method, order_status
- order_id 중복: 0
- 결측치: 없음
- 데이터 기간: 2025-07-09 ~ 2026-07-08
- order_date는 날짜 형태로 보이지만 원본에서는 문자열로 저장되어 있음

4. order_items
- 크기: 764행 × 5열
- 컬럼: order_item_id, order_id, product_id, quantity, unit_price
- 결측치: 없음
- order_id는 한 주문에 여러 상품이 포함될 수 있어 반복될 수 있음

확인한 파일 간 관계는 다음과 같습니다.

- customers.customer_id → orders.customer_id
- orders.order_id → order_items.order_id
- products.product_id → order_items.product_id

위 세 관계에서 연결되지 않는 ID는 현재 확인 결과 0건

이 정보를 바탕으로 다음 내용을 검토해 주세요.

1. 현재 컬럼과 파일 간 관계만으로 실제 수행 가능한 분석 질문을 4~5개 제안해 주세요.
2. 각 분석 질문마다 필요한 데이터셋, 주요 컬럼, 필요한 병합 관계를 함께 적어 주세요.
3. 매출과 관련된 분석을 제안할 경우 cancelled 또는 refunded 주문을 어떻게 처리할지는 별도의 기준이 필요하다는 점을 구분해 주세요.
4. 현재 제공한 정보만으로 확인할 수 있는 사실과 추가 검증이 필요한 가정을 구분해 주세요.
5. 데이터에 없는 원인이나 고객 행동을 추측해서 단정하지 말아 주세요.

#6. LLM 분석 아이디어 검증

LLM은 다음 5가지 분석을 제안하였다.

1. 월별 주문금액 및 주문 건수 추이  
orders의 order_date를 기준으로 월별 주문 건수를 집계하고, orders.order_id와 order_items.order_id를 연결하여 quantity × unit_price로 월별 주문금액을 계산하는 분석이다. 단, cancelled, refunded 주문을 포함할지 여부에 따라 결과의 의미가 달라질 수 있다. 

2. 카테고리별·상품별 판매량 및 주문금액  
order_items와 products를 product_id로 연결하여 상품별 또는 카테고리별 판매 수량과 주문금액을 비교하는 분석이다. 이를 통해 어떤 상품군의 판매 비중이 높은지 확인할 수 있다. 

3. 연령대·성별·지역별 구매 지표  
customers의 age, gender, city와 orders, order_items를 customer_id, order_id로 연결하여 고객 그룹별 주문 빈도와 누적 주문금액 차이를 비교하는 분석이다. 그룹별 차이는 확인할 수 있지만 그 차이의 원인까지 현재 데이터만으로 단정할 수는 없다. 

4. 결제수단별 주문 상태 분포  
orders의 payment_method와 order_status를 이용하여 결제수단별 주문 건수와 완료·취소·환불 비율을 비교하는 분석이다. 두 컬럼이 모두 orders에 있으므로 바로 수행할 수 있다. 

5. 가입 후 첫 구매까지 소요 기간  
customers.signup_date와 고객별 최초 orders.order_date를 비교하여 가입 후 첫 주문까지 걸린 기간을 계산하는 분석이다. 이를 위해 customer_id로 두 데이터를 연결하고, signup_date를 날짜 타입으로 변환해야 한다. 또한 취소 주문을 첫 구매로 볼지 여부는 별도의 기준이 필요하다.

실제 데이터의 컬럼과 관계를 확인한 결과, 위 분석에 필요한 주요 컬럼은 모두 존재했고  
관계를 검증한 결과 아래의 관계를 만족하지 않아 연결되지 않는 경우는 없었다.  
customers.customer_id → orders.customer_id  
orders.order_id → order_items.order_id  
products.product_id → order_items.product_id  
따라서 5개 분석 아이디어는 데이터 구조상 모두 수행 가능한 것으로 판단하였다.

다만 월별 주문금액, 카테고리별 주문금액, 고객별 구매금액 분석에서는 cancelled, refunded 주문을 어떻게 처리할지 기준이 필요하다.
또한 가입 후 첫 구매 기간 분석에서는 signup_date의 날짜 타입 변환과 취소 주문을 첫 구매로 볼지 여부를 추가로 정의해야 한다.

결론적으로 LLM의 분석 아이디어는 대부분 활용 가능했지만 실제 분석에 적용하기 전에 컬럼 존재 여부, 연결 관계, 주문 상태 처리 기준을 직접 검증해야 한다고 판단하였다.


# Chapter 03 제출 답안 양식. 데이터의 첫인상 읽기

> 주 제출물은 실행 완료 Notebook `chapter03/chapter03.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름:김희철
- GitHub ID:KHeeCheol
- 작성일:2026-09-23
- 최종 제출 URL:https://github.com/KHeeCheol/llm-data-analysis-study/blob/main/notebooks/chapter03.ipynb

## 1. 데이터 로딩과 구조 확인
### 실행/결과
- 4개 CSV 로딩 여부: 모두 정상적으로 DataFrame으로 로딩됨
- 각 데이터 shape:
  - customers: 150행 × 6열
  - products: 100행 × 4열
  - orders: 300행 × 5열
  - order_items: 764행 × 5열
- 주요 컬럼:  
  - customers: customer_id, name, gender, age, city, signup_date
  - products: product_id, product_name, category, price
  - orders: order_id, customer_id, order_date, payment_method, order_status
  - order_items: order_item_id, order_id, product_id, quantity, unit_price
- dtypes에서 주목한 컬럼:  
  - customer_id, product_id, order_id 등 ID 컬럼은 숫자형으로 저장되어 있었음
  - age, price, quantity, unit_price는 실제 수치 분석에 사용할 수 있는 숫자형 컬럼으로 확인
  - signup_date와 order_date는 날짜 형태의 값이지만 문자열로 저장되어 있었음

### Evidence
![데이터 구조 확인](images/step01_structure.png)

### 결과 관찰
4개 CSV 파일은 모두 정상적으로 DataFrame으로 로딩되었다.  
가장 행 수가 많은 데이터셋은 order_items로 764행이었으며, orders의 300행보다 많았다. 하나의 주문에 여러 상품이 포함될 수 있어 한 주문이 여러 행으로 구성될 수 있기 때문으로 보인다.


### 나의 해석과 판단
분석 전에는 각 컬럼의 데이터 타입뿐 아니라 실제 의미를 확인하는 것이 중요하다고 판단하였다.  
예를 들어 age와 customer_id는 모두 숫자형이지만 age는 수치해석이 가능한 값인 반면 customer_id는 고객을 구분하기 위한 식별자이므로 동일한 방식으로 해석하면 안 된다. 또한 signup_date와 order_date는 날짜처럼 보이지만 문자열로 저장되어 있었기 때문에 분석 전에 날짜 타입으로 변환하는 과정이 필요하였다.


### 업무·분석적 의미
구조를 먼저 확인하지 않고 분석을 시작할 때 생길 수 있는 문제를 작성하세요.
데이터 구조를 확인하지 않고 바로 분석을 시작하면 ID 컬럼을 일반 숫자형 변수로 잘못 분석할 수도 있고, 문자열로 저장된 날짜를 그대로 사용하여 기간 계산에서 오류가 발생할 수 있다. 또한 여러 파일을 함께 분석하려면 어떤 컬럼이 다른 파일과 연결되는 키인지 먼저 확인해야 한다.

### 한계와 추가 확인 사항
컬럼명, 데이터 타입만으로는 데이터 품질이 정상인지 판단할 수 없다.
결측치와 중복데이터, 주요 ID의 고유성, 숫자형 값의 범위, 범주형 값의 표기 일관성, 날짜 변환 가능여부와 파일간 키 관계는 별도로 확인할 필요가 있다.

## 2. 결측·중복·키 품질
- 주요 ID 결측: 0건
- 주요 ID 중복:  
  - customers.customer_id: 0건
  - products.product_id: 0건
  - orders.order_id: 0건
  - order_items.order_id: 464건
- 전체 행 중복: 0건

![결측 중복 점검](images/step02_quality.png)

### 결과 관찰
4개 데이터셋의 모든 컬럼에서 결측치는 확인되지 않았으며 전체 행이 완전히 동일한 중복 데이터도 0건이었다.
고객, 상품, 주문을 구분하는 customer_id, product_id, order_id의 중복은 모두 0건이었다.
반면 order_items의 order_id는 464건의 중복이 확인되었다.

### 나의 해석과 판단
customers.customer_id, products.product_id, orders.order_id는 각각 고객, 상품, 주문을 구분하는 식별자이므로 중복이 없는 것이 정상적인 결과라고 판단하였다.
반면 order_items.order_id는 주문 상세 행을 구분하는 고유 ID가 아니라 각 상품 항목이 어느 주문에 속하는지를 나타내는 연결용 ID이다.
하나의 주문에 여러상품이 포함될 수 있으므로 order_id가 반복되는 것은 데이터 오류라기보다 1:N 관계에 따른 정상적인 구조로 판단하였다.

### 업무·분석적 의미
결측치나 중복을 처리할때 단순히 개수만 확인하고 일괄 삭제하면 정상적인 거래 정보를 제거할 수 있다.
특히 주문 상세 데이터처럼 하나의 주문에 여러 상품이 연결되는 구조에서는 연결용 ID의 반복이 정상적으로 발생할 수 있으므로 컬럼의 역할을 함께 고려해야 한다.

### 한계와 추가 확인 사항
현재는 결측치, 전체 행 중복, 주요 ID의 중복 여부를 확인하였다. 하지만 값 자체의 오류나 범주형 값의 표기 불일치, 비정상적인 숫자 범위 등은 중복 검사만으로 확인할 수 없으므로 별도의 점검이 필요하다.

## 3. 숫자형·범주형·날짜 점검
- 숫자형 범위에서 주목한 값:  
  - age: 19 ~ 69
  - price: 5,000 ~ 200,000
  - quantity: 1 ~ 5
  - unit_price: 5,000 ~ 200,000  
- 범주형 빈도에서 주목한 값:  
  - city: 10개 값
  - category: 7개 값
  - payment_method: 4개 값
  - order_status: completed 184건, cancelled 64건, refunded 52건  
- 날짜 변환 실패 건수:  
 - order_date: 0건
- 날짜 범위:  
  - 2025-07-09 ~ 2026-07-08

![기본 분포와 날짜 확인](images/step03_distribution.png)

### 결과 관찰
age는 19~69세, quantity는 1~5개의 범위를 보였다. 상품 가격과 주문 상세 단가는 모두 5,000원 ~ 200,000원 범위였다.
order_status는 completed가 184건으로 가장 많았고 cancelled 64건, refunded 52건으로 확인되었다.
order_date는 문자열 데이터타입 이었지만 날짜 타입으로 변환한 결과 실패 건수는 0건이었으며 데이터 기간은 2025년 7월 9일부터 2026년 7월 8일까지였다.

### 나의 해석과 판단
이상해 보이는 값이 실제 오류인지 업무적으로 가능한 값인지 구분하기 위해 무엇을 더 확인해야 하는지 작성하세요.
현재 확인한 최소/최대값에서는 즉시 오류라고 판단할 정도로 비정상적인 값은 보이지 않았다. 하지만 order_status의 completed, cancelled, refunded는 이후 주문금액이나 매출을 분석할 때 결과에 직접 영향을 줄 수 있으므로 어떤 상태를 포함할지 기준을 먼저 정할 필요가 있다.

### 업무·분석적 의미
숫자형 데이터의 범위와 범주형 값의 분포를 먼저 확인하면 잘못 입력된 값이나 분석 기준에 크게 영향을 줄 수 있는 범주를 초기에 발견할 수 있다.
또한 날짜를 str 상태로 그대로 사용하면 월별 또는 기간별 분석에서 오류가 발생할 수 있으므로 날짜 타입으로 변환하고 변환 실패 여부를 확인하는 과정이 필요하다.

### 한계와 추가 확인 사항
현재는 기본 통계와 빈도를 중심으로 확인하였다. 실제 이상치 여부를 판단하려면 업무 기준이나 분포를 추가로 확인해야 한다. 추가로 signup_date도 분석에 사용할 경우 날짜 타입으로 변환할 필요가 있다. 

## 4. CSV 간 키 관계 검증
- 없는 `customer_id`: 0건
- 없는 `order_id`: 0건
- 없는 `product_id`: 0건 

![PK FK 관계 검증](images/step04_relationship.png)

### 결과 관찰
다음 세 관계를 확인한 결과 연결되지 않는 ID는 모두 0건이었다.  
- customers.customer_id → orders.customer_id  
- orders.order_id → order_items.order_id  
- products.product_id → order_items.product_id  
따라서 현재 데이터에서는 주문이 존재하지 않는 고객 ID나 주문상세에서 존재하지 않는 주문 ID 및 상품 ID를 참조하는 경우는 확인되지 않았다.

### 나의 해석과 판단
데이터 타입이나 공백 이슈, 일부 기준 데이터의 누락 등 여러 원인이 있을 수 있으므로 먼저 발생 원인을 확인해야 한다.

### 업무·분석적 의미
여러 데이터를 병합하여 분석할 때 키 관계가 맞지 않으면 일부 주문이나 상품이 누락되어 집계 결과가 달라질 수 있다. 따라서 병합 전 키관계를 검증하는 것은 이후 분석 결과의 신뢰성을 확보하기 위한 기본사항이라 생각하였다.

### 한계와 추가 확인 사항
현재는 ID가 다른 파일에 존재하는지만 확인하였다. 키가 존재하더라도 실제로 의미가 올바르게 연결되어 있는지까지 보장하는 것은 아니므로 필요한 경우 병합 후 행 수 변화나 일부 샘플 데이터를 추가로 확인할 필요가 있다.

## 5. LLM 구조 설명 검증
- LLM에 제공한 Safe Context:  
  - 4개 데이터셋의 행/열 크기
  - 컬럼명
  - 결측치 여부
  - 주문 데이터 기간
  - 파일 간 주요 키 관계
  - 연결되지 않는 ID가 0건이라는 검증 결과

- LLM이 제안한 추가 점검:  
  - 월별 주문금액 및 주문 건수 추이
  - 카테고리별·상품별 판매량 및 주문금액
  - 연령대·성별·지역별 구매 지표
  - 결제수단별 주문 상태 분포
  - 가입 후 첫 구매까지 소요 기간
- 실제 데이터에서 확인한 항목:  
  - 제안된 분석에 필요한 주요 컬럼이 존재함
  - 세 가지 주요 파일 간 키 관계에서 연결되지 않는 ID가 없음
  - order_status에 completed, cancelled, refunded가 존재함
  - order_date는 날짜형으로 변환 가능하며 변환 실패가 없음
- 채택/수정/보류한 내용:
  - 5개 분석 아이디어는 데이터 구조상 수행 가능하므로 채택
  - 주문금액을 실제 매출로 해석하기 전 cancelled, refunded 처리 기준은 추가 정의 필요
  - 가입 후 첫 구매 분석에서는 signup_date 변환과 첫 구매의 정의를 추가 확인  

![LLM 구조 검토](images/step05_llm.png)

### 나의 해석과 판단
LLM 제안 중 가장 유용했던 것과 가장 조심해야 할 것을 작성하세요.
LLM이 제안한 분석 아이디어는 현재 데이터 구조와 대체로 일치하였지만, 제안 내용을 그대로 사용하는 것보다 실제 컬럼과 키 관계를 확인한 뒤 적용하는 과정이 필요하다고 판단하였다.
특히 매출이나 첫 구매와 같은 개념은 데이터 컬럼만으로 기준이 자동으로 정해지는 것이 아니므로 분석 목적에 맞게 업무 기준을 먼저 정의할 필요가 있다.

### 한계와 추가 확인 사항
LLM은 제공된 구조 정보만을 바탕으로 분석 방향을 제안하므로 실제 데이터 값이나 업무 기준까지 자동으로 알 수 없다.
따라서 LLM의 제안은 분석 후보를 정리하는 데 활용하고 실제 적용 여부는 직접 실행한 결과를 기준으로 검증해야 한다.

## 6. Chapter 03 최종 판단
### 데이터의 첫인상 3가지
1.4개 CSV에서 결측치와 전체 행 중복은 확인되지 않았고 주요 ID와 파일간 키 관계도 기본적으로 정상적으로 연결되어 있었다.
2.customers, orders, order_items, products는 각각 고객, 주문, 주문 상세, 상품 정보를 가지고 있고 customer_id, order_id, product_id를 이용해 서로 연결할 수 있는 구조였다.
3.기본적인 데이터 품질은 양호해 보였지만 날짜 타입 변환이나 cancelled, refunded 주문의 처리처럼 실제 분석 전에 별도의 기준을 정해야 하는 항목이 존재했다.

### 다음 Chapter 전에 반드시 확인/처리해야 할 항목
1.signup_date와 같이 문자열로 저장된 날짜 컬럼을 실제 분석 목적에 맞게 날짜 타입으로 변환한다
2.주문금액이나 매출 분석 전에 completed, cancelled, refunded의 포함 기준을 정의한다.
3.여러 데이터셋을 병합한 뒤 행수와 집계 결과가 예상한 구조와 일치하는지 다시 확인한다.

### 현재 데이터만으로 단정할 수 없는 것
현재 데이터에서는 고객 특성, 상품 카테고리, 결제수단 등에 따른 차이를 분석할 수 있지만 그 차이가 발생한 원인까지 현재 데이터만으로 단정할 수는 없다.
또한 quantity × unit_price 로 주문금액을 계산할 수는 있지만 이를 실제 매출로 정의하기 위해서는 취소/환불 처리기준과 업무상 매출정의를 추가로 확인할 필요가 있다.
## 최종 제출 체크
- [O] Notebook을 처음부터 끝까지 실행했습니다.
- [O] 오류 셀이 남아 있지 않습니다.
- [O] 핵심 Evidence를 첨부했습니다.
- [O] 관찰과 해석을 구분했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter03/chapter03.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.

## 마무리

이번 장의 핵심은 “데이터를 불러왔다”에서 끝내지 않는 것입니다.

데이터 분석을 시작하기 전에 파일, 행과 열, 컬럼명, 데이터 타입, 결측치, 중복, 키 관계를 확인해야 이후 분석이 흔들리지 않습니다. 다음 장에서는 이 구조를 바탕으로 pandas의 선택, 필터링, 정렬, 집계 기초를 다룹니다.
